# Data Ingestion Method 1: CREATE TABLE AS SELECT (CTAS)

## Overview
CTAS is a SQL statement that creates a new table from the results of a SELECT query. It's the simplest and most straightforward method for ingesting data from existing sources into Delta Lake.

## Key Characteristics
- **Best for**: One-time data loads or initial data setup
- **Performance**: Fast for large data transfers
- **Schema Inference**: Automatic schema detection
- **Incremental Support**: No - full table replacement
- **Simplicity**: Easiest to use


## Prerequisites

Ensure you have:
1. Access to the data source (CSV, Parquet, JSON, etc.)
2. Appropriate catalog and schema created
3. Read permissions on the source data


## Step 1: Read and Explore Source Data

We'll load customer data that represents an initial bulk load scenario - perfect for CTAS use cases.

In [ ]:
-- Preview the raw CSV data
SELECT * FROM csv.`/Volumes/data_analysis/sample_data/customers.csv` LIMIT 5

In [ ]:
-- Using read_files for better control
SELECT * FROM read_files(
  '/Volumes/data_analysis/sample_data/customers.csv',
  format=>'csv'
) LIMIT 5

## Step 2: Create Delta Table Using CTAS

This creates a managed Delta table with cleaned and renamed columns.


In [ ]:
-- Drop table if exists (for demo purposes)
DROP TABLE IF EXISTS customers_delta_ctas;

-- Create table using CTAS with column transformation
CREATE TABLE customers_delta_ctas AS 
SELECT 
  customer_id,
  customer_name,
  email,
  country,
  region,
  customer_tier,
  signup_date,
  CURRENT_TIMESTAMP() as ingestion_timestamp
FROM read_files(
  '/Volumes/data_analysis/sample_data/customers.csv',
  format=>'csv'
)

## Step 3: Verify the Created Table


In [ ]:
-- Check row count
SELECT COUNT(*) as total_rows FROM customers_delta_ctas

In [ ]:
-- View table schema and metadata
DESCRIBE EXTENDED customers_delta_ctas

In [ ]:
-- View sample data
SELECT * FROM customers_delta_ctas LIMIT 10

## Step 4: CTAS with Partitioning

For large tables, partitioning can improve query performance.


In [ ]:
DROP TABLE IF EXISTS customers_partitioned;

-- CTAS with partitioning by region for better query performance
CREATE TABLE customers_partitioned
USING DELTA
PARTITIONED BY (region)
AS
SELECT 
  customer_id,
  customer_name,
  email,
  country,
  region,
  customer_tier,
  signup_date,
  CURRENT_TIMESTAMP() as ingestion_timestamp
FROM read_files(
  '/Volumes/data_analysis/sample_data/customers.csv',
  format=>'csv'
)

In [ ]:
-- Verify partitioning
SHOW PARTITIONS customers_partitioned

## Advantages of CTAS

✅ **Simple Syntax**: Easy to understand and implement

✅ **Atomic Operation**: Table is fully created or not at all (no partial data)

✅ **Schema Inference**: Automatic data type detection

✅ **Data Transformation**: Can apply transformations during ingestion

✅ **Partitioning Support**: Can partition data during table creation

✅ **Performance**: Single operation, no multiple writes


## Disadvantages of CTAS

❌ **No Incremental Loads**: Full table replacement each time

❌ **No Idempotency**: Must manually handle duplicate runs

❌ **Resource Intensive**: Full table recreation uses significant resources

❌ **Downtime**: Original table is unavailable during recreation


## Use Cases for CTAS

1. **Initial Data Load**: Migrating from external system to Databricks
2. **Snapshot Creation**: Creating point-in-time copies of data
3. **Data Warehouse Setup**: Building star/snowflake schemas
4. **One-Time Analysis**: Loading external datasets for analysis
5. **Data Transformation**: Complex transformations on load


## Summary

CTAS is the most straightforward method for data ingestion, ideal for:
- Initial data loads
- Complete table replacements
- Simple transformation logic

For incremental updates and streaming data, consider **COPY INTO** or **Auto Loader** instead.
